In [ ]:
import numpy as np
from scipy.stats import unitary_group

import random
from random import choices

from functools import partial
from itertools import product, combinations
from opt_einsum import contract

import jax
import jax.numpy as jnp

import time
import os

import tensorcircuit as tc
import qutip as qt


import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import rc
rc('text', usetex=True)
rc('font', size=25)
rc('axes', linewidth=3)
rc('text.latex', preamble=r'\usepackage{amsfonts}')

In [1]:
def IsingHamiltonian(N, hx0, hz0, J_mat):
    Xs = []
    Zs = []
    for i in range(N):
        X = qt.tensor([qt.sigmax() if k==i else qt.qeye(2) for k in range(N)])
        Xs.append(X)
        Z = qt.tensor([qt.sigmaz() if k==i else qt.qeye(2) for k in range(N)])
        Zs.append(Z)
    
    H = 0
    for i in range(N):
        H += hx0[i] * Xs[i]
        H += hz0[i] * Zs[i]
    
    J_graph = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            if i < j: J_graph[i, j] = 1
    J_mat = J_mat * J_graph

    for i in range(N):
        for j in range(N):
            H = H + J_mat[i, j] * Zs[i] * Zs[j]

    return H

In [ ]:
def channel_pure_ancilla(U, Na, Nb):
    # construct quantum channel where B starts in |0> and is traced out
    K_list = []
    Ks = U.reshape(2**Na, 2**Nb, 2**Na, 2**Nb)[:, :, :, 0]
    Ks = [Ks[:, m] for m in range(2**Nb)]
    return np.stack(Ks)

def channel_mix_ancilla(U, Na, Nb):
    # construct quantum channel where B starts in I/d and is traced out
    K_list = []
    Ks = U.reshape(2**Na, 2**Nb, 2**Na, 2**Nb)
    Ks = [Ks[:, i, :, j] for i in range(2**Nb) for j in range(2**Nb)]
    return np.stack(Ks)/np.sqrt(2**Nb)

def superoperator(Ks):
    E = 0
    for i in range(len(Ks)):
        E += np.kron(Ks[i].conj(), Ks[i])
    return E

In [ ]:
def HolevoInfo(rho1, rho2):
    rho_avg = (rho1 + rho2) * 0.5

    val_avg = np.linalg.eigvalsh(rho_avg)
    val_avg = jnp.where(val_avg > 0, val_avg, 1e-14)
    S_avg = -np.sum(val_avg * np.log2(val_avg))

    v1 = np.linalg.eigvalsh(rho1)
    v1 = jnp.where(v1 > 0, v1, 1e-14)
    S1 = -np.sum(v1 * np.log2(v1))

    v2 = np.linalg.eigvalsh(rho2)
    v2 = jnp.where(v2 > 0, v2, 1e-14)
    S2 = -np.sum(v2 * np.log2(v2))

    return S_avg - 0.5*(S1 + S2)

In [ ]:
N_all = 10

rseed = 17462 # seed for generator so get repeatable CE string
rs = np.random.RandomState(seed = rseed) # random number generator

hx0_vec_all = 1.0 + 0.3*rs.randn(N_all)
hz0_vec_all = 0.0 + 0.3*rs.randn(N_all)
# hz1_vec = 1.0 + 0.3*rs.randn(N_all)

J_mat_all = 0 + 1.0*rs.rand(N_all, N_all)

In [ ]:
N_system_list = np.array([1, 2, 3, 4, 5, 6])

In [ ]:
# hamiltonian
N_ancilla = 3
t_evo = 50

vals_list = []
vecs_list = []
for r in range(len(N_system_list)):
    t0 = time.time()
    N_system = N_system_list[r]
    N_qubit = N_system + N_ancilla

    H0 = IsingHamiltonian(N_qubit, hx0_vec_all[:N_qubit], hz0_vec_all[:N_qubit], J_mat_all[:N_qubit, :N_qubit])
    U0  = (- 1j * t_evo * H0).expm()

    Ks = channel_pure_ancilla(U0.full(), N_system, N_ancilla)
    E = superoperator(Ks)

    vals, vecs = sp.linalg.eig(E)
    
    vals_list.append(vals)
    vecs_list.append(vecs)
    print(f'N={N_system}, time={time.time() - t0:.3f}')
    
np.savez(f'data/mem_Volterra/channelVal_Ising_Nb{N_ancilla}tau{t_evo}', *vals_list)
np.savez(f'data/mem_Volterra/channelVec_Ising_Nb{N_ancilla}tau{t_evo}', *vecs_list)

In [ ]:
# hamiltonian holevo information
vals_list = np.load(f'data/mem_Volterra/channelVal_Ising_Nb{N_ancilla}tau{t_evo}.npz')
vals_list = [vals_list[f'arr_{i}'] for i in range(len(N_system_list))]
vecs_list = np.load(f'data/mem_Volterra/channelVec_Ising_Nb{N_ancilla}tau{t_evo}.npz')
vecs_list = [vecs_list[f'arr_{i}'] for i in range(len(N_system_list))]

ts = np.arange(21)
chi = np.zeros((len(N_system_list), len(ts)))

for r in range(len(N_system_list)):
    N_system = N_system_list[r]
    
    order = np.argsort(np.abs(vals_list[r]))[::-1]

    rho_fix = vecs_list[r][:, order[0]].reshape(2**N_system, 2**N_system)
    rho_fix /= np.trace(rho_fix)

    sigma1 = vecs_list[r][:, order[1]].reshape(2**N_system, 2**N_system)
    if np.abs(np.imag(vals_list[r][order[1]])) > 1e-14:
        rho1 = rho_fix + 0.05*(sigma1 + sigma1.conj().T)
    else:
        rho1 = rho_fix + 0.05*sigma1
 
    for i in range(len(ts)):
        if np.abs(np.imag(vals_list[r][order[1]])) > 1e-14:
            chi[r, i] = HolevoInfo(rho_fix, rho_fix + 0.05*(vals_list[r][order[1]]**ts[i]*sigma1 
                                                            + vals_list[r][order[2]]**ts[i]*sigma1.conj().T))
        else:
            chi[r, i] = HolevoInfo(rho_fix, rho_fix + 0.05* np.abs(vals_list[r][order[1]])**ts[i] * sigma1)
np.save(f'data/mem_Volterra/CQHinfo_Ising_Nb{N_ancilla}tau{t_evo}.npy', chi)

In [ ]:
# haar + pure
N_ancilla = 2

vals_list = []
vecs_list = []
for r in range(len(N_system_list)):
    t0 = time.time()
    N_system = N_system_list[r]
    N_qubit = N_system + N_ancilla

    U0 = qt.rand_unitary_haar(2**N_qubit, seed=2025)

    Ks = channel_pure_ancilla(U0.full(), N_system, N_ancilla)
    E = superoperator(Ks)

    vals, vecs = sp.linalg.eig(E)
    
    vals_list.append(vals)
    vecs_list.append(vecs)
    print(f'N={N_system}, time={time.time() - t0:.3f}')
    
np.savez(f'data/mem_Volterra/channelVal_haar_Nb{N_ancilla}', *vals_list)
np.savez(f'data/mem_Volterra/channelVec_haar_Nb{N_ancilla}', *vecs_list)

In [ ]:
# haar with fully mixed state
N_ancilla = 2

vals_list = []
vecs_list = []
for r in range(len(N_system_list)):
    t0 = time.time()
    N_system = N_system_list[r]
    N_qubit = N_system + N_ancilla

    U0 = qt.rand_unitary_haar(2**N_qubit, seed=2025)

    Ks = channel_mix_ancilla(U0.full(), N_system, N_ancilla)
    E = superoperator(Ks)

    vals, vecs = sp.linalg.eig(E)
    
    vals_list.append(vals)
    vecs_list.append(vecs)
    print(f'N={N_system}, time={time.time() - t0:.3f}')
    
np.savez(f'data/mem_Volterra/channelVal_haarMix_Nb{N_ancilla}', *vals_list)
np.savez(f'data/mem_Volterra/channelVec_haarMix_Nb{N_ancilla}', *vecs_list)